In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
import os
import time
import copy
import pickle
from tqdm import tqdm # Import tqdm for progress bars

# --- Configuration ---
NUM_CLIENTS = 10
NUM_ROUNDS = 50
LEARNING_RATE = 0.01
BATCH_SIZE = 64
K_FIXED = 5 # Fixed number of local epochs
ALPHA_CHALLENGING = 0.1 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

print(f"Running on device: {DEVICE}")

Running on device: cuda


### Step 1: Model and Data Preparation (Unchanged)

In [20]:
# --- Model Definition for CIFAR-10 ---
class CIFAR_CNN(nn.Module):
    def __init__(self):
        super(CIFAR_CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=5, padding=2)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5, padding=2)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(8 * 8 * 32, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 8 * 8 * 32)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# --- Data Partitioning Function ---
def dirichlet_partition_data(dataset, num_clients, alpha, batch_size):
    num_classes = len(dataset.classes)
    class_indices = [np.where(np.array(dataset.targets) == i)[0] for i in range(num_classes)]
    client_indices = [[] for _ in range(num_clients)]
    for c_indices in class_indices:
        proportions = np.random.dirichlet([alpha] * num_clients)
        num_samples_for_class = (proportions * len(c_indices)).astype(int)
        rem = len(c_indices) - num_samples_for_class.sum()
        for i in range(rem):
            num_samples_for_class[i % num_clients] += 1
        start = 0
        for i in range(num_clients):
            end = start + num_samples_for_class[i]
            client_indices[i].extend(c_indices[start:end])
            start = end
    client_loaders = []
    for indices in client_indices:
        client_dataset = Subset(dataset, indices)
        loader = DataLoader(client_dataset, batch_size=batch_size, shuffle=True)
        client_loaders.append(loader)
    return client_loaders

# --- Data Loading and Transformation ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])
full_train_dataset = datasets.CIFAR10('./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

### Step 2: Client Training and Evaluation Functions

In [21]:
def evaluate_model(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            outputs = model(data)
            _, predicted = torch.max(outputs.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    return 100 * correct / total

def train_client_fedavg(model, data_loader, lr, local_epochs):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(local_epochs):
        for data, target in data_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
    return model.state_dict()

def train_client_scaffold(model, data_loader, lr, local_epochs, c_global_dict, c_local_dict):
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    initial_model_state = copy.deepcopy(model.state_dict())
    
    # Calculate c_diff = c_global - c_local for the gradient correction term
    c_diff = {name: c_global_dict[name] - c_local_dict[name] for name in c_global_dict}

    for epoch in range(local_epochs):
        for data, target in data_loader:
            data, target = data.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            for name, param in model.named_parameters():
                if param.grad is not None:
                    param.grad.data += c_diff[name].to(DEVICE)
            optimizer.step()
    
    final_model_state = model.state_dict()
    num_steps = len(data_loader) * local_epochs
    
    # Calculate y_delta: the change in model parameters
    y_delta = {name: final_model_state[name].cpu() - initial_model_state[name].cpu() for name in final_model_state}
    
    # Calculate new local control variate (c_plus) and its change (c_delta)
    c_plus_dict = copy.deepcopy(c_local_dict)
    c_delta_dict = {}
    with torch.no_grad():
        for name in c_global_dict:
            update_term = (initial_model_state[name].cpu() - final_model_state[name].cpu()) / (num_steps * lr)
            c_plus_dict[name] = c_local_dict[name] - c_global_dict[name] + update_term
            c_delta_dict[name] = c_plus_dict[name] - c_local_dict[name]
            
    # Return deltas as per the reference implementation
    return y_delta, c_delta_dict, c_plus_dict

### Step 3: Self-Contained Experiment Runner Functions

In [22]:
def run_fedavg_experiment(alpha, num_rounds, num_clients, local_epochs, batch_size, lr, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    result_filename = f'task4_2/results/fedavg_alpha_{alpha}.pkl'
    os.makedirs(os.path.dirname(result_filename), exist_ok=True)

    if os.path.exists(result_filename):
        print(f"\n--- Loading existing FedAvg results from {result_filename} ---")
        with open(result_filename, 'rb') as f:
            return pickle.load(f)['accuracies']

    print(f"\n--- Running FedAvg Experiment (alpha={alpha}) ---")
    client_loaders = dirichlet_partition_data(full_train_dataset, num_clients, alpha, batch_size)
    global_model = CIFAR_CNN().to(DEVICE)
    accuracies = []

    for round_num in range(num_rounds):
        start_time = time.time()
        local_updates = []
        client_iterator = tqdm(range(num_clients), desc=f"FedAvg Round {round_num+1}/{num_rounds}", leave=False)
        for client_idx in client_iterator:
            local_model = CIFAR_CNN().to(DEVICE)
            local_model.load_state_dict(global_model.state_dict())
            updated_state = train_client_fedavg(local_model, client_loaders[client_idx], lr, local_epochs)
            local_updates.append(updated_state)
        
        global_state_dict = global_model.state_dict()
        for key in global_state_dict.keys():
            global_state_dict[key] = torch.stack([s[key] for s in local_updates]).mean(dim=0)
        global_model.load_state_dict(global_state_dict)
        
        acc = evaluate_model(global_model, test_loader)
        accuracies.append(acc)
        round_time = time.time() - start_time
        print(f"FedAvg Round {round_num+1}/{num_rounds} | Accuracy: {acc:.2f}% | Time: {round_time:.2f}s")
    
    print(f"--- Saving FedAvg results to {result_filename} ---")
    with open(result_filename, 'wb') as f:
        pickle.dump({'accuracies': accuracies}, f)
    
    return accuracies

def run_scaffold_experiment(alpha, num_rounds, num_clients, local_epochs, batch_size, lr, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    result_filename = f'task4_2/results/scaffold_alpha_{alpha}.pkl'
    os.makedirs(os.path.dirname(result_filename), exist_ok=True)

    if os.path.exists(result_filename):
        print(f"\n--- Loading existing SCAFFOLD results from {result_filename} ---")
        with open(result_filename, 'rb') as f:
            return pickle.load(f)['accuracies']

    print(f"\n--- Running SCAFFOLD Experiment (alpha={alpha}) ---")
    client_loaders = dirichlet_partition_data(full_train_dataset, num_clients, alpha, batch_size)
    global_model = CIFAR_CNN().to(DEVICE)
    c_global_dict = {name: torch.zeros_like(param).cpu() for name, param in global_model.named_parameters()}
    c_clients = {i: copy.deepcopy(c_global_dict) for i in range(num_clients)}
    accuracies = []

    # This experiment assumes full client participation (S=N)
    S_clients = num_clients

    for round_num in range(num_rounds):
        start_time = time.time()
        y_delta_cache = []
        c_delta_cache = []
        client_iterator = tqdm(range(num_clients), desc=f"SCAFFOLD Round {round_num+1}/{num_rounds}", leave=False)
        for client_idx in client_iterator:
            local_model = CIFAR_CNN().to(DEVICE)
            local_model.load_state_dict(global_model.state_dict())
            y_delta, c_delta, c_plus = train_client_scaffold(
                model=local_model, data_loader=client_loaders[client_idx],
                lr=lr, local_epochs=local_epochs,
                c_global_dict=c_global_dict, c_local_dict=c_clients[client_idx])
            y_delta_cache.append(y_delta)
            c_delta_cache.append(c_delta)
            c_clients[client_idx] = c_plus # Update the client's control state

        # --- Server Aggregation ---
        # Update global model by averaging model deltas
        global_state_dict = global_model.state_dict()
        for name in global_state_dict:
            avg_y_delta = torch.stack([d[name] for d in y_delta_cache]).mean(dim=0)
            global_state_dict[name] += avg_y_delta.to(DEVICE)
        global_model.load_state_dict(global_state_dict)
        
        # Update global control variate with the delta rule
        for name in c_global_dict:
            avg_c_delta = torch.stack([d[name] for d in c_delta_cache]).mean(dim=0)
            c_global_dict[name] += (S_clients / num_clients) * avg_c_delta

        acc = evaluate_model(global_model, test_loader)
        accuracies.append(acc)
        round_time = time.time() - start_time
        print(f"SCAFFOLD Round {round_num+1}/{num_rounds} | Accuracy: {acc:.2f}% | Time: {round_time:.2f}s")

    print(f"--- Saving SCAFFOLD results to {result_filename} ---")
    with open(result_filename, 'wb') as f:
        pickle.dump({'accuracies': accuracies}, f)
        
    return accuracies

### Step 4: Plotting and Main Execution

In [23]:
def plot_comparison(fedavg_accuracies, scaffold_accuracies, alpha):
    figure_filename = f'task4_2/figures/fedavg_vs_scaffold_alpha_{alpha}.png'
    os.makedirs(os.path.dirname(figure_filename), exist_ok=True)

    plt.figure(figsize=(12, 8))
    plt.style.use('seaborn-v0_8-whitegrid')

    rounds = range(1, len(fedavg_accuracies) + 1)
    plt.plot(rounds, fedavg_accuracies, marker='x', linestyle='--', label=f'FedAvg (Baseline)')
    plt.plot(rounds, scaffold_accuracies, marker='o', linestyle='-', label=f'SCAFFOLD')

    plt.title(f'FedAvg vs. SCAFFOLD on CIFAR-10 (α={alpha})', fontsize=16)
    plt.xlabel("Communication Round", fontsize=12)
    plt.ylabel("Global Test Accuracy (%)", fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.xticks(np.arange(0, len(rounds) + 1, 5))
    plt.yticks(np.arange(0, 101, 10))
    plt.tight_layout()
    plt.savefig(figure_filename)
    print(f"\nPlot saved to {figure_filename}")
    plt.show()

# --- Main Execution Block ---
if __name__ == '__main__':
    # --- Run FedAvg ---
    # To skip running FedAvg, just comment out the line below
    # fedavg_accuracies = run_fedavg_experiment(
    #     alpha=ALPHA_CHALLENGING, num_rounds=NUM_ROUNDS, num_clients=NUM_CLIENTS,
    #     local_epochs=K_FIXED, batch_size=BATCH_SIZE, lr=LEARNING_RATE, seed=SEED)
    
    # --- Run SCAFFOLD ---
    # To skip running SCAFFOLD, just comment out the line below
    scaffold_accuracies = run_scaffold_experiment(
        alpha=ALPHA_CHALLENGING, num_rounds=NUM_ROUNDS, num_clients=NUM_CLIENTS,
        local_epochs=K_FIXED, batch_size=BATCH_SIZE, lr=LEARNING_RATE, seed=SEED)

    # --- Plot the results ---
    # if 'fedavg_accuracies' in locals() and 'scaffold_accuracies' in locals():
    #     plot_comparison(fedavg_accuracies, scaffold_accuracies, ALPHA_CHALLENGING)
    # else:
    #     print("\nSkipping plotting as one or both experiments were not run.")


--- Running SCAFFOLD Experiment (alpha=0.1) ---


SCAFFOLD Round 1/50 | Accuracy: 16.70% | Time: 61.93s


SCAFFOLD Round 2/50 | Accuracy: 18.15% | Time: 38.84s


SCAFFOLD Round 3/50 | Accuracy: 23.69% | Time: 39.94s


SCAFFOLD Round 4/50 | Accuracy: 32.54% | Time: 40.07s


...

SCAFFOLD Round 46/50 | Accuracy: 56.52% | Time: 40.65s


SCAFFOLD Round 47/50 | Accuracy: 57.12% | Time: 40.09s


SCAFFOLD Round 48/50 | Accuracy: 57.46% | Time: 40.50s


SCAFFOLD Round 49/50 | Accuracy: 57.82% | Time: 40.35s


SCAFFOLD Round 50/50 | Accuracy: 55.50% | Time: 40.58s
--- Saving SCAFFOLD results to task4_2/results/scaffold_alpha_0.1.pkl ---
